In [ ]:
# ── UPDATE THESE THREE PATHS FOR YOUR SYSTEM ──────────────────
HDR_PATH   = r"C:\Users\PESU-RF\Desktop\Capstone Team - 88\f250812t01p00r14rfl\f250812t01p00r14_rfl.hdr"
IMG_PATH   = r"C:\Users\PESU-RF\Desktop\Capstone Team - 88\f250812t01p00r14rfl\f250812t01p00r14_rfl"
OUTPUT_DIR = r"C:\Users\PESU-RF\Desktop\Capstone Team - 88\aviris_dbscan_outputs"
# ─────────────────────────────────────────────────────────────

ROW_START    = 0
ROW_END      = None
PATCH_SIZE   = 7
SAMPLE_LIMIT = 1_000_000
TEST_SPLIT   = 0.2
RANDOM_SEED  = 42

# ── DBSCAN parameters ─────────────────────────────────────────
# EPS      : neighbourhood radius in spectral space
#            smaller = more, tighter clusters
#            larger  = fewer, looser clusters
# MIN_SAMPLES : minimum pixels to form a core point
#               higher = stricter, fewer clusters, more noise
# These will be tuned in Step 7 using the k-distance graph
EPS         = 0.5    # starting value — Step 7 will guide you
MIN_SAMPLES = 50     # starting value
DBSCAN_FIT_SIZE = 500_000   # pixels to fit DBSCAN on (full scene too large)

In [2]:
import sys
!{sys.executable} -m pip install spectral tensorly psutil scikit-learn

import os
import numpy as np
import matplotlib.pyplot as plt
import spectral.io.envi as envi
import tensorly as tl
import psutil
from sklearn.model_selection import train_test_split
from sklearn.cluster import DBSCAN


HDR_PATH   = HDR_PATH.strip()
IMG_PATH   = IMG_PATH.strip()
OUTPUT_DIR = OUTPUT_DIR.strip()
os.makedirs(OUTPUT_DIR, exist_ok=True)

ram = psutil.virtual_memory()
print(f"System RAM  : {ram.total/1e9:.1f} GB total  |  {ram.available/1e9:.1f} GB available")
print(f"Estimated patch size for {SAMPLE_LIMIT:,} pixels : "
      f"{SAMPLE_LIMIT * PATCH_SIZE * PATCH_SIZE * 175 * 4 / 1e9:.2f} GB")

for label, path in [("HDR", HDR_PATH), ("IMG", IMG_PATH)]:
    if os.path.exists(path):
        print(f"  [{label}] FOUND  ({os.path.getsize(path)/1e6:.1f} MB)")
    else:
        raise FileNotFoundError(f"{label} not found: {path}")

In [ ]:
print("STEP 1 — Loading DS")
img         = envi.open(HDR_PATH, IMG_PATH)
wavelengths = np.array(img.bands.centers)

print(f"  Full image size  : {img.shape}  (rows, cols, bands)")

if ROW_END is None:
    ROW_END = img.shape[0]

mmap   = img.open_memmap(writeable=False)
subset = np.array(mmap[ROW_START:ROW_END, :, :], dtype=np.float32)
del mmap

subset = np.where(subset < -0.5, 0.0, subset)
subset = np.clip(subset, 0.0, 1.0)

print(f"  Loaded subset    : {subset.shape}")
print(f"  Wavelength range : {wavelengths[0]:.1f} - {wavelengths[-1]:.1f} nm")
print(f"  Value range      : [{subset.min():.4f}, {subset.max():.4f}]")
print(f"  Memory used      : {subset.nbytes / 1e9:.2f} GB")

In [ ]:
print("\n" + "-"*55)
print("STEP 2 — Visualize")
print("-"*55)

def find_band(wl, nm):
    return int(np.argmin(np.abs(wl - nm)))

def pct_stretch(arr, lo=2, hi=98):
    lo_v, hi_v = np.percentile(arr, [lo, hi])
    return np.clip((arr - lo_v) / (hi_v - lo_v + 1e-8), 0, 1)

vis = subset[:min(500, subset.shape[0])]
b_nir, b_red = find_band(wavelengths, 850), find_band(wavelengths, 660)
b_grn, b_blu = find_band(wavelengths, 550), find_band(wavelengths, 460)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].imshow(pct_stretch(vis[:, :, [b_red, b_grn, b_blu]]))
axes[0, 0].set_title("True color (R-G-B)"); axes[0, 0].axis("off")
axes[0, 1].imshow(pct_stretch(vis[:, :, [b_nir, b_red, b_grn]]))
axes[0, 1].set_title("False color (NIR-R-G)"); axes[0, 1].axis("off")
im = axes[1, 0].imshow(vis[:, :, b_nir], cmap="gray")
plt.colorbar(im, ax=axes[1, 0], fraction=0.046)
axes[1, 0].set_title("NIR band (850 nm)"); axes[1, 0].axis("off")
px = subset[min(250, subset.shape[0]-1), 400, :]
axes[1, 1].plot(wavelengths, px, color="forestgreen", lw=1.5)
axes[1, 1].axvspan(1340, 1450, alpha=0.2, color="red",    label="Water abs. 1")
axes[1, 1].axvspan(1800, 1970, alpha=0.2, color="orange", label="Water abs. 2")
axes[1, 1].axvspan(2450, 2600, alpha=0.2, color="gray",   label="Sensor noise")
axes[1, 1].set_xlabel("Wavelength (nm)"); axes[1, 1].set_ylabel("Reflectance")
axes[1, 1].set_title("Spectral signature (reflectance)"); axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
plt.suptitle("AVIRIS Reflectance Overview (f250812)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_overview.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("\n" + "-"*55)
print("STEP 3 — Bad band removal")
print("-"*55)

good_mask = (
    ((wavelengths >= 400)  & (wavelengths < 1340)) |
    ((wavelengths > 1450)  & (wavelengths < 1800)) |
    ((wavelengths > 2000)  & (wavelengths <= 2450))
)
good_idx        = np.where(good_mask)[0]
wavelengths_ok  = wavelengths[good_idx]

refl_184        = subset[:, :, good_idx].copy()
wavelengths_184 = wavelengths[good_idx]

data = refl_184.copy()
del subset

print(f"  Bands kept      : {good_idx.size} / {len(wavelengths)}")
print(f"  Data shape      : {data.shape}")
print(f"  Value range     : [{data.min():.4f}, {data.max():.4f}]  ← already reflectance")

In [ ]:
print("\n" + "-"*55)
print("STEP 4 — Clip & fix values")
print("-"*55)

data = np.clip(data, 0, 1)
data = np.nan_to_num(data, nan=0.0, posinf=1.0, neginf=0.0)

print(f"  Range : [{data.min():.4f}, {data.max():.4f}]  ← clipped to [0, 1]")
print(f"  NaN   : {np.isnan(data).any()}   Inf : {np.isinf(data).any()}")

In [ ]:
print("\n" + "-"*55)
print("STEP 5 — Variance-based band filtering")
print("-"*55)

rows, cols, n_bands = data.shape
band_var   = data.reshape(-1, n_bands).var(axis=0)
threshold  = np.percentile(band_var, 5)
keep_mask  = band_var > threshold

data           = data[:, :, keep_mask]
wavelengths_ok = wavelengths_ok[keep_mask]

refl_184_filt        = refl_184[:, :, keep_mask]
wavelengths_184_filt = wavelengths_184[keep_mask]

print(f"  Bands after filter : {data.shape[2]}  (removed {keep_mask.size - keep_mask.sum()})")

plt.figure(figsize=(10, 3))
plt.bar(range(n_bands), band_var, width=1, color="steelblue", alpha=0.7)
plt.axhline(threshold, color="red", linestyle="--", label=f"threshold ({threshold:.4f})")
plt.xlabel("Band index"); plt.ylabel("Variance")
plt.title("Per-band variance — bands below red line dropped")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "02_band_variance.png"), dpi=150, bbox_inches="tight")
plt.show()
print("  Saved -> 02_band_variance.png")

In [ ]:
print("\n" + "-"*55)
print("STEP 6 — Per-band Z-score normalization")
print("-"*55)

rows, cols, nb = data.shape
X_flat    = data.reshape(-1, nb).astype(np.float32)

band_mean = X_flat.mean(axis=0)
band_std  = X_flat.std(axis=0)
band_std  = np.where(band_std < 1e-8, 1e-8, band_std)

X_norm    = (X_flat - band_mean) / band_std
data_norm = X_norm.reshape(rows, cols, nb)

data_norm = np.clip(data_norm, -5.0, 5.0)
print(f"  After outlier clip : [{data_norm.min():.3f}, {data_norm.max():.3f}]")

del X_flat, X_norm, data

print(f"  Shape  : {data_norm.shape}")
print(f"  Mean   : {data_norm.mean():.6f}  (should be ~0)")
print(f"  Std    : {data_norm.std():.6f}   (should be ~1)")
print(f"  Memory : {data_norm.nbytes / 1e9:.2f} GB")

np.savez(
    os.path.join(OUTPUT_DIR, "band_metadata.npz"),
    wavelengths = wavelengths_ok,
    band_mean   = band_mean,
    band_std    = band_std
)
print("  Saved band_metadata.npz ✓")
print("\n  ✓ Skipping DOS — data is already surface reflectance")

In [ ]:
print("\n" + "-"*55)
print("STEP 7 — k-distance graph (tune EPS for DBSCAN)")
print("-"*55)

# ── Why we need this ──────────────────────────────────────────
# DBSCAN needs eps: the neighbourhood radius.
# Too small → everything is noise.
# Too large → everything merges into one cluster.
# The k-distance graph finds the 'knee' — the natural eps.
#
# Method: for each point, compute distance to its k-th nearest
# neighbour (k = MIN_SAMPLES). Sort distances. The knee of the
# curve is the optimal eps.

from sklearn.neighbors import NearestNeighbors

flat_refl  = refl_184_filt.reshape(-1, refl_184_filt.shape[2]).astype(np.float32)
valid_mask = flat_refl.max(axis=1) > 0.01
valid_idx  = np.where(valid_mask)[0]

print(f"  Valid pixels : {len(valid_idx):,}")

# Sample for k-distance (10k is enough to see the knee)
rng        = np.random.default_rng(RANDOM_SEED)
kdist_idx  = rng.choice(valid_idx, size=min(10_000, len(valid_idx)), replace=False)
kdist_sample = flat_refl[kdist_idx]

print(f"  Computing {MIN_SAMPLES}-NN distances on {len(kdist_idx):,} sample pixels ...")
nbrs = NearestNeighbors(n_neighbors=MIN_SAMPLES, metric='euclidean', n_jobs=-1)
nbrs.fit(kdist_sample)
distances, _ = nbrs.kneighbors(kdist_sample)
k_distances  = np.sort(distances[:, -1])[::-1]   # distance to k-th neighbour, sorted desc

plt.figure(figsize=(10, 4))
plt.plot(k_distances, color='steelblue', lw=1.5)
plt.axhline(EPS, color='red', linestyle='--', linewidth=2,
            label=f'Current EPS = {EPS}')
plt.xlabel('Points sorted by distance (descending)')
plt.ylabel(f'{MIN_SAMPLES}-NN distance')
plt.title('k-Distance Graph — look for the KNEE to set EPS')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "kdistance_plot.png"), dpi=150)
plt.show()

print(f"\n  Distance statistics:")
print(f"    5th  percentile : {np.percentile(k_distances, 5):.4f}")
print(f"    25th percentile : {np.percentile(k_distances, 25):.4f}")
print(f"    50th percentile : {np.percentile(k_distances, 50):.4f}")
print(f"    75th percentile : {np.percentile(k_distances, 75):.4f}")
print(f"\n  → Look at the plot. Find the knee (sharpest bend).")
print(f"    Set EPS in Cell 0 to that value, then re-run Step 8.")
print(f"    Current EPS = {EPS}")

In [ ]:
print("\n" + "-"*55)
print("STEP 8 — DBSCAN clustering on reflectance")
print("-"*55)

# ── Why DBSCAN is different from GMM ─────────────────────────
# GMM: assumes Gaussian blobs, needs N_CLUSTERS specified upfront
# DBSCAN: finds arbitrarily shaped clusters, discovers N_CLUSTERS
# automatically, marks outlier pixels as noise (-1 label)
#
# Problem: DBSCAN is O(n²) in memory — cannot run on 9M pixels.
# Solution (standard practice):
#   1. Fit DBSCAN on a representative sample (50k pixels)
#   2. Compute cluster centroids from fitted sample
#   3. Assign all remaining pixels to nearest centroid
#      (nearest centroid = fast, O(n) assignment)
# This is called the "DBSCAN + nearest centroid" approach.

from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA as skPCA

flat_refl  = refl_184_filt.reshape(-1, refl_184_filt.shape[2]).astype(np.float32)
valid_mask = flat_refl.max(axis=1) > 0.01
valid_idx  = np.where(valid_mask)[0]

print(f"  Total pixels  : {len(flat_refl):,}")
print(f"  Valid pixels  : {len(valid_idx):,}")
print(f"  DBSCAN fit on : {DBSCAN_FIT_SIZE:,} sample pixels")

# ── Step 1: Sample and reduce dims for DBSCAN ─────────────────
# PCA to 20 dims before DBSCAN — speeds up distance computation
# significantly and removes noise dimensions
rng        = np.random.default_rng(RANDOM_SEED)
fit_idx    = rng.choice(valid_idx, size=min(DBSCAN_FIT_SIZE, len(valid_idx)), replace=False)
fit_sample = flat_refl[fit_idx]

print(f"  Reducing to 20 PCA dims for DBSCAN speed ...")
pca_db = skPCA(n_components=20, random_state=RANDOM_SEED)
fit_pca = pca_db.fit_transform(fit_sample)
print(f"  PCA explained variance : {pca_db.explained_variance_ratio_.sum()*100:.1f}%")

# ── Step 2: Fit DBSCAN on sample ──────────────────────────────
print(f"  Fitting DBSCAN (eps={EPS}, min_samples={MIN_SAMPLES}) ...")
db = DBSCAN(
    eps         = EPS,
    min_samples = MIN_SAMPLES,
    metric      = 'euclidean',
    n_jobs      = -1
)
db_labels = db.fit_predict(fit_pca)   # -1 = noise

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise    = (db_labels == -1).sum()
N_CLUSTERS = n_clusters

print(f"  Clusters found  : {n_clusters}")
print(f"  Noise points    : {n_noise:,}  ({n_noise/len(db_labels)*100:.1f}% of sample)")

if n_clusters == 0:
    raise ValueError(
        "DBSCAN found 0 clusters. EPS is too small or too large.\n"
        "Check the k-distance plot and adjust EPS in Cell 0."
    )
if n_clusters > 20:
    print(f"  WARNING: {n_clusters} clusters is a lot — consider increasing EPS.")
if n_noise / len(db_labels) > 0.3:
    print(f"  WARNING: >30% noise — consider decreasing EPS or MIN_SAMPLES.")

# ── Step 3: Compute centroids from non-noise sample points ────
# Centroid = mean reflectance of all pixels assigned to that cluster
cluster_ids  = sorted([c for c in set(db_labels) if c >= 0])
centroids    = np.zeros((n_clusters, fit_pca.shape[1]), dtype=np.float32)
centroids_rf = np.zeros((n_clusters, flat_refl.shape[1]), dtype=np.float32)

for i, cid in enumerate(cluster_ids):
    mask = db_labels == cid
    centroids[i]    = fit_pca[mask].mean(axis=0)        # PCA space centroid
    centroids_rf[i] = fit_sample[mask].mean(axis=0)     # reflectance centroid
    print(f"    Cluster {cid} : {mask.sum():>6,} sample pixels")

# ── Step 4: Assign ALL valid pixels to nearest centroid ───────
# This extends DBSCAN labels to the full scene
print(f"\n  Assigning {len(valid_idx):,} valid pixels to nearest centroid ...")
BATCH       = 100_000
all_labels  = np.zeros(len(valid_idx), dtype=np.int32)

for start in range(0, len(valid_idx), BATCH):
    end        = min(start + BATCH, len(valid_idx))
    batch_rf   = flat_refl[valid_idx[start:end]]        # (B, bands)
    batch_pca  = pca_db.transform(batch_rf)             # (B, 20)

    # Distance from each pixel to each centroid: (B, n_clusters)
    diffs  = batch_pca[:, np.newaxis, :] - centroids[np.newaxis, :, :]  # (B, K, 20)
    dists  = np.linalg.norm(diffs, axis=2)              # (B, K)
    all_labels[start:end] = dists.argmin(axis=1)        # nearest centroid
    print(f"    {end:,} / {len(valid_idx):,}", end="\r")

print(f"    Done.                    ")

labels_flat            = np.zeros(rows * cols, dtype=np.int32)
labels_flat[valid_idx] = all_labels + 1   # 0 = unlabelled
class_map              = labels_flat.reshape(rows, cols)

print(f"\n  Final pixel distribution:")
colors = plt.cm.tab10.colors
for i in range(N_CLUSTERS):
    count = (class_map == i+1).sum()
    bar   = "█" * min(40, count // max(1, class_map.size // 400))
    print(f"    [{i}]  {count:>8,}  {bar}")

# Plot cluster mean spectra (from reflectance centroids)
fig, ax = plt.subplots(figsize=(13, 5))
for i in range(N_CLUSTERS):
    ax.plot(wavelengths_184_filt, centroids_rf[i],
            color=colors[i%10], lw=1.5, label=f"Cluster {i}")
ax.axvspan(1340, 1450, alpha=0.08, color="red",    label="Water abs. 1")
ax.axvspan(1800, 1970, alpha=0.08, color="orange", label="Water abs. 2")
ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Reflectance")
ax.set_title(f"DBSCAN cluster centroids — {N_CLUSTERS} clusters  "
             f"(eps={EPS}, min_samples={MIN_SAMPLES})")
ax.legend(fontsize=8, ncol=3); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "03_cluster_spectra.png"), dpi=150, bbox_inches="tight")
plt.show()
print("  Saved → 03_cluster_spectra.png")

In [ ]:
print("\n" + "-"*55)
print("STEP 9 — Pixel selection from DBSCAN")
print("-"*55)

CLASS_NAMES = [f"Class_{i}" for i in range(N_CLUSTERS)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im = axes[0].imshow(class_map, cmap="tab10",
                    vmin=0, vmax=N_CLUSTERS, interpolation="none")
plt.colorbar(im, ax=axes[0], label="Cluster (0=unassigned)")
axes[0].set_title(f"DBSCAN: {N_CLUSTERS} clusters  "
                  f"(eps={EPS}, min_samples={MIN_SAMPLES})")
counts = [int((class_map == i+1).sum()) for i in range(N_CLUSTERS)]
colors = plt.cm.tab10.colors
axes[1].bar(range(N_CLUSTERS), counts, color=list(colors[:N_CLUSTERS]))
axes[1].set_xticks(range(N_CLUSTERS))
axes[1].set_xticklabels(CLASS_NAMES, rotation=40, ha="right", fontsize=8)
axes[1].set_ylabel("Pixel count")
axes[1].set_title("Pixels per DBSCAN cluster")
axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_classification_map.png"), dpi=150, bbox_inches="tight")
plt.show()

lf           = class_map.flatten()
labelled_idx = np.where(lf > 0)[0]
labels_all   = lf[labelled_idx] - 1   # 0-indexed

rng = np.random.default_rng(RANDOM_SEED)
if len(labelled_idx) > SAMPLE_LIMIT:
    chosen = []
    for cls in range(N_CLUSTERS):
        cls_idx = np.where(labels_all == cls)[0]
        n_take  = max(1, int(SAMPLE_LIMIT * len(cls_idx) / len(labelled_idx)))
        if len(cls_idx) > 0:
            sel = rng.choice(cls_idx, size=min(n_take, len(cls_idx)), replace=False)
            chosen.extend(sel.tolist())
    chosen     = np.array(sorted(chosen))
    pixel_idx  = labelled_idx[chosen]
    labels_sel = labels_all[chosen]
else:
    pixel_idx  = labelled_idx
    labels_sel = labels_all

print(f"  pixel_idx  : {len(pixel_idx):,} pixels")
print(f"  labels_sel : {labels_sel.shape}  classes: {np.unique(labels_sel)}")

np.savez(os.path.join(OUTPUT_DIR, "class_info.npz"),
         class_names=np.array(CLASS_NAMES, dtype=object),
         n_classes=N_CLUSTERS, method="dbscan")
print("  Saved class_info.npz ✓")
print("""
  NOTE: Class names are Class_0 to Class_N for now.
  Run Step 9b (SAM matching) to replace with USGS mineral names.
""")

In [ ]:
# STEP 9b — USGS splib07 SAM matching
print("\n" + "-"*55)
print("STEP 9b — USGS splib07 SAM matching")
print("-"*55)

import os
import numpy as np

LIBRARY_DIR = r"C:\Users\PESU-RF\Desktop\Capstone Team - 88\usgs_splib07\ASCIIdata\ASCIIdata_splib07b_cvAVIRISc2014"
CHAPTERS    = ["ChapterM_Minerals", "ChapterS_SoilsAndMixtures"]

wl_file = r"C:\Users\PESU-RF\Desktop\Capstone Team - 88\usgs_splib07\ASCIIdata\ASCIIdata_splib07b_cvAVIRISc2014\s07_AV14_Wavelengths_in_microns_224_ch_AVIRIS14a.txt"

sensor_wl = []
with open(wl_file, errors="ignore") as f:
    for i, line in enumerate(f):
        if i == 0: continue
        line = line.strip()
        if not line: continue
        try:
            val = float(line)
            sensor_wl.append(val * 1000)
        except ValueError:
            continue
sensor_wl = np.array(sensor_wl)
print(f"  Sensor wavelengths : {len(sensor_wl)} bands")
print(f"  Range : {sensor_wl[0]:.1f} – {sensor_wl[-1]:.1f} nm")

def load_chapter(chapter_dir, sensor_wl, target_wl):
    names, spectra = [], []
    txt_files = sorted([f for f in os.listdir(chapter_dir) if f.endswith(".txt")])
    for fname in txt_files:
        refl = []
        with open(os.path.join(chapter_dir, fname), errors="ignore") as f:
            for i, line in enumerate(f):
                if i == 0: continue
                line = line.strip()
                if not line: continue
                try: refl.append(float(line))
                except ValueError: continue
        if len(refl) < 10: continue
        refl    = np.array(refl, dtype=np.float32)
        refl    = np.where(refl < -0.1, np.nan, refl)
        min_len = min(len(refl), len(sensor_wl))
        refl    = refl[:min_len]
        wl_use  = sensor_wl[:min_len]
        valid   = ~np.isnan(refl)
        if valid.sum() < 10: continue
        refl    = np.clip(refl[valid], 0, 1)
        wl_use  = wl_use[valid]
        interp  = np.interp(target_wl, wl_use, refl, left=np.nan, right=np.nan)
        if np.isnan(interp).any(): continue
        parts   = fname.replace(".txt", "").split("_")
        names.append(parts[2] if len(parts) > 2 else fname)
        spectra.append(interp.astype(np.float32))
    return names, spectra

all_names, all_spectra = [], []
for chapter in CHAPTERS:
    chapter_path = os.path.join(LIBRARY_DIR, chapter)
    n, s = load_chapter(chapter_path, sensor_wl, wavelengths_184_filt)
    all_names.extend(n)
    all_spectra.extend(s)
    print(f"  Loaded {len(n):>4} spectra from {chapter}")

if len(all_spectra) == 0:
    raise ValueError("No spectra loaded — check LIBRARY_DIR path")

lib_spectra = np.array(all_spectra, dtype=np.float32)
print(f"\n  Total library spectra : {len(all_names)}")

# SAM matching with unique assignment
def sam_angle(a, b):
    cos = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

# Use reflectance centroids for SAM matching
CLASS_NAMES = []
used_names  = set()

print("\n  Cluster → Best USGS mineral match:")
print(f"  {'Cluster':<8} {'Pixels':>12}  {'Angle':>7}  Mineral")
print("  " + "-"*60)

for i in range(N_CLUSTERS):
    centre = centroids_rf[i]   # reflectance centroid from DBSCAN
    angles = [sam_angle(centre, lib_spectra[j]) for j in range(len(lib_spectra))]

    # Pick best unused mineral
    sorted_idx = np.argsort(angles)
    for best_idx in sorted_idx:
        candidate = all_names[best_idx]
        if candidate not in used_names:
            break

    name  = all_names[best_idx]
    angle = angles[best_idx]
    count = int((class_map == i+1).sum())
    CLASS_NAMES.append(name)
    used_names.add(name)
    print(f"  [{i}]  {count:>12,}  {angle:>6.1f}°  {name}")

np.savez(
    os.path.join(OUTPUT_DIR, "class_info.npz"),
    class_names = np.array(CLASS_NAMES, dtype=object),
    n_classes   = N_CLUSTERS,
    method      = "dbscan+usgs_splib07_sam"
)
print(f"\n  CLASS_NAMES = {CLASS_NAMES}")
print("  Saved class_info.npz with USGS mineral names ✓")

In [ ]:
print("\n" + "-"*55)
print("STEP 10 — Patch extraction")
print("-"*55)
rows, cols, nb = data_norm.shape

patch_gb = len(pixel_idx) * PATCH_SIZE * PATCH_SIZE * nb * 4 / 1e9
print(f"  Pixels to extract : {len(pixel_idx):,}")
print(f"  Patch array size  : {patch_gb:.2f} GB")

if patch_gb > psutil.virtual_memory().available / 1e9 * 0.7:
    print(f"  WARNING: May exceed available RAM. Reduce SAMPLE_LIMIT.")

print(f"\n  Padding image ...")
pad        = PATCH_SIZE // 2
padded     = np.pad(data_norm, ((pad, pad), (pad, pad), (0, 0)), mode="reflect")
pixel_rows = pixel_idx // cols
pixel_cols = pixel_idx  % cols

print(f"  Extracting {len(pixel_idx):,} patches ({PATCH_SIZE}×{PATCH_SIZE}×{nb}) ...")
patches = np.empty((len(pixel_idx), PATCH_SIZE, PATCH_SIZE, nb), dtype=np.float32)

for i, (r, c) in enumerate(zip(pixel_rows, pixel_cols)):
    patches[i] = padded[r : r + PATCH_SIZE, c : c + PATCH_SIZE, :]
    if (i + 1) % 10_000 == 0:
        print(f"    {i+1:,} / {len(pixel_idx):,} done", end="\r")

del padded

print(f"\n  Done.")
print(f"  Patches shape  : {patches.shape}")
print(f"  Memory used    : {patches.nbytes / 1e9:.2f} GB")
print(f"  NaN?           : {np.isnan(patches).any()}")

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np

print("\n" + "-"*55)
print("STEP 11 — Train/test split & save (memory safe)")
print("-"*55)

# Remove rare classes before splitting
class_counts = np.bincount(labels_sel, minlength=N_CLUSTERS)
rare_classes  = np.where(class_counts < 2)[0]
if len(rare_classes) > 0:
    print(f"  Removing {len(rare_classes)} rare class(es): {rare_classes.tolist()}")
    keep        = ~np.isin(labels_sel, rare_classes)
    patches     = patches[keep]
    labels_sel  = labels_sel[keep]
    pixel_idx   = pixel_idx[keep]
    unique_cls  = np.unique(labels_sel)
    remap       = {old: new for new, old in enumerate(unique_cls)}
    labels_sel  = np.array([remap[l] for l in labels_sel], dtype=np.int32)
    CLASS_NAMES = [CLASS_NAMES[i] for i in unique_cls]
    N_CLUSTERS  = len(unique_cls)
    print(f"  Active classes : {N_CLUSTERS}")

sss = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SPLIT, random_state=RANDOM_SEED)
train_idx, test_idx = next(sss.split(np.zeros(len(labels_sel)), labels_sel))

y_train = labels_sel[train_idx]
y_test  = labels_sel[test_idx]

print(f"  Train : {len(train_idx):,} patches")
print(f"  Test  : {len(test_idx):,} patches")

print("  Saving train patches ...")
train_out = np.lib.format.open_memmap(
    os.path.join(OUTPUT_DIR, "patches_train.npy"),
    mode='w+', dtype='float32',
    shape=(len(train_idx), PATCH_SIZE, PATCH_SIZE, patches.shape[3])
)
for i, idx in enumerate(train_idx):
    train_out[i] = patches[idx]
    if (i+1) % 50_000 == 0:
        print(f"    {i+1:,} / {len(train_idx):,}", end="\r")
del train_out
print(f"    Done.                    ")

print("  Saving test patches ...")
test_out = np.lib.format.open_memmap(
    os.path.join(OUTPUT_DIR, "patches_test.npy"),
    mode='w+', dtype='float32',
    shape=(len(test_idx), PATCH_SIZE, PATCH_SIZE, patches.shape[3])
)
for i, idx in enumerate(test_idx):
    test_out[i] = patches[idx]
    if (i+1) % 50_000 == 0:
        print(f"    {i+1:,} / {len(test_idx):,}", end="\r")
del test_out
print(f"    Done.                    ")

np.save(os.path.join(OUTPUT_DIR, "labels_train.npy"), y_train)
np.save(os.path.join(OUTPUT_DIR, "labels_test.npy"),  y_test)
np.save(os.path.join(OUTPUT_DIR, "pixel_idx.npy"),    pixel_idx)
np.save(os.path.join(OUTPUT_DIR, "dbscan_centroids.npy"), centroids_rf)
np.savez(
    os.path.join(OUTPUT_DIR, "class_info.npz"),
    class_names = np.array(CLASS_NAMES, dtype=object),
    n_classes   = N_CLUSTERS,
    method      = "dbscan+usgs_splib07_sam"
)

print(f"\n  Saved to {OUTPUT_DIR}")
print(f"  Train class distribution:")
for i, name in enumerate(CLASS_NAMES):
    count = (y_train == i).sum()
    print(f"    [{i}] {name:<30}  {count:>6,}")

X_train = np.load(os.path.join(OUTPUT_DIR, "patches_train.npy"), mmap_mode='r')
X_test  = np.load(os.path.join(OUTPUT_DIR, "patches_test.npy"),  mmap_mode='r')
print(f"\n  X_train : {X_train.shape}  X_test : {X_test.shape}")

In [ ]:
print("\n" + "-"*55)
print("STEP 12 — Tensor formatting for TTN")
print("-"*55)

N_tr, P, _, B = X_train.shape
N_te          = X_test.shape[0]

X_train_4d = X_train
X_test_4d  = X_test

print(f"  Train tensor : {X_train_4d.shape}  dtype: {X_train_4d.dtype}")
print(f"  Test  tensor : {X_test_4d.shape}   dtype: {X_test_4d.dtype}")
print(f"  Train labels : {y_train.shape}  classes: {np.unique(y_train)}")
print(f"  Test  labels : {y_test.shape}")
print(f"\n  Memory used  : {X_train_4d.nbytes / 1e9:.1f} GB")
print("\n  → X_train_4d and y_train ready for TTN model.")

In [ ]:
print("\n" + "="*55)
print("        PREPROCESSING COMPLETE (DBSCAN+SAM)")
print("="*55)
print(f"  Source file              : f250812t01p00r14_rfl  (reflectance)")
print(f"  Rows loaded              : {ROW_END - ROW_START}")
print(f"  After bad band removal   : {good_idx.size} bands")
print(f"  After variance filter    : {nb} bands")
print(f"  Pixels extracted         : {len(pixel_idx):,}")
print(f"  Patches shape            : {patches.shape}")
print(f"  Active classes           : {N_CLUSTERS}")
print(f"  Class names              : {CLASS_NAMES}")
print(f"  DBSCAN eps               : {EPS}")
print(f"  DBSCAN min_samples       : {MIN_SAMPLES}")
print(f"  Train tensor (Format A)  : {X_train_4d.shape}")
print(f"  Test  tensor (Format A)  : {X_test_4d.shape}")

chunk_size = 100_000
has_nan    = False
for start in range(0, len(patches), chunk_size):
    if np.isnan(patches[start:start+chunk_size]).any():
        has_nan = True
        break
print(f"  NaN in patches?          : {has_nan}")

p_min, p_max = np.inf, -np.inf
for start in range(0, len(patches), chunk_size):
    chunk = patches[start:start+chunk_size]
    p_min = min(p_min, float(chunk.min()))
    p_max = max(p_max, float(chunk.max()))
print(f"  Value range              : [{p_min:.3f}, {p_max:.3f}]")
print("="*55)
print(f"  Outputs → {OUTPUT_DIR}")
print("="*55)
print("""
  KEY DIFFERENCES vs GMM VERSION:
  ─────────────────────────────────────────────────────
  Clustering  : DBSCAN (density-based) instead of GMM
  N_CLUSTERS  : Auto-discovered (not specified upfront)
  Noise       : DBSCAN noise pixels excluded via
                nearest-centroid assignment
  Cluster shape: Arbitrary (not Gaussian assumption)
  Eps tuning  : k-distance graph (Step 7)
  Centroids   : Mean of sample pixels (not model params)
  SAM matching: Same USGS library, same unique assignment
  ─────────────────────────────────────────────────────
  X_train_4d + y_train → ready for TTN model
""")